# Day 3 — Pandas: Handling Missing Values, Merging, Apply

In [21]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

## Part 1 — Handling Missing Values

In [22]:
#First always assess the damage
print(df.isnull().sum())
print(df.isnull().sum()/len(df) * 100) #as percentage

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cabin          77.104377
Embarked        0.224467
dtype: float64


3 Strategies depending on context:

---

In [23]:
#Strategy 1 - Drop rows (only when very few missing)
df_dropped = df.dropna(subset=['Embarked']) # Only 2 missing - safe to drop
print(df_dropped.shape)

(889, 12)


In [24]:
#Strategy 2 - Fill with a statistic (numeric columns)
median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)

# Why median not mean? Mean is skewed by outliers. Median is more robust.
print(df['Age'].isnull().sum()) # Should be 0

0


In [25]:
#Strategy 3 - Fill with most frequent value (categorical)
most_common_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(most_common_embarked)

In [26]:
#Strategy 4 - Drop the whole column (when too much is missing)
print(df['Cabin'].isnull().sum()) # 687 out of 891 - unusable
df = df.drop(columns=['Cabin'])

687


In [27]:
#Verify everything is clean
print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


#### Rule of thumb:

Less than 5% missing → drop rows,
5–30% missing → fill with median/mode,
More than 50% missing → drop the column

## Part 2 - Apply & Map

In [43]:
# .map() - simple value replacement(categorical encoding)
df['Sex_encoded'] = df['Sex'].map({'male': 0, 'female': 1})
print(df[['Sex', 'Sex_encoded']].head())

      Sex  Sex_encoded
0    male            0
1  female            1
2  female            1
3  female            1
4    male            0


In [29]:
# .apply() with a lambda - inline function
df['Fare_log'] = df['Fare'].apply(lambda x: np.log1p(x))

#log transform reduces the effect of extreme outliers in Fare
print(df[['Fare', 'Fare_log']].head(10))

      Fare  Fare_log
0   7.2500  2.110213
1  71.2833  4.280593
2   7.9250  2.188856
3  53.1000  3.990834
4   8.0500  2.202765
5   8.4583  2.246893
6  51.8625  3.967694
7  21.0750  3.094446
8  11.1333  2.495954
9  30.0708  3.436268


In [30]:
# .apply() with a proper function - when logic is complex

def categorise_age(age):
    if age<18:
        return 'Child'
    elif age<60:
        return 'Adult'
    else:
        return 'Senior'

df['AgeGroup'] = df['Age'].apply(categorise_age)

#Notice: no NaN problem now because age null were filled in part 1
print(df['AgeGroup'].value_counts())

AgeGroup
Adult     752
Child     113
Senior     26
Name: count, dtype: int64


In [45]:
# Apply across rows (axis=1) - when need multiple columns
def is_alone(row):
    return 'Yes' if (row['SibSp'] + row['Parch']) == 0 else 'No'

df['IsAlone'] = df.apply(is_alone, axis=1)
#were solo travelars more or less likely survived?

## Part 3 - Merging DataFrames

In [33]:
# Create two small DataFrames to practice on
passengers = pd.DataFrame({
    'PassengerID': [1,2,3,4,5],
    'Name' : ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve'],
    'Pclass': [1,2,3,1,2]})

ticket_info = pd.DataFrame({
    'PassengerID': [1,2,3,6],  #Note: 4,5 missing, 6 is extra
    'Fare': [100.0, 50.0, 10.0, 75.0]})

In [34]:
# Inner Join - only rows that exist in Both
inner = pd.merge(passengers, ticket_info, on='PassengerID', how='inner')
print(inner) # only 1,2,3

   PassengerID     Name  Pclass   Fare
0            1    Alice       1  100.0
1            2      Bob       2   50.0
2            3  Charlie       3   10.0


In [35]:
# Left Join - all rows from left, matched from right (NaN if no match)
left = pd.merge(passengers, ticket_info, on='PassengerID', how='left')
print(left) # all  passengers, Fare NaN for 4 and 5

   PassengerID     Name  Pclass   Fare
0            1    Alice       1  100.0
1            2      Bob       2   50.0
2            3  Charlie       3   10.0
3            4    Diana       1    NaN
4            5      Eve       2    NaN


In [36]:
# Right Join - opposite of left
right = pd.merge(passengers, ticket_info, on='PassengerID', how='right')
print(right) # includes passenger 6 from ticket info

   PassengerID     Name  Pclass   Fare
0            1    Alice     1.0  100.0
1            2      Bob     2.0   50.0
2            3  Charlie     3.0   10.0
3            6      NaN     NaN   75.0


In [37]:
# Outer Join - everything from both, NaN where no match
outer = pd.merge(passengers, ticket_info, on='PassengerID', how='outer')
print(outer)

   PassengerID     Name  Pclass   Fare
0            1    Alice     1.0  100.0
1            2      Bob     2.0   50.0
2            3  Charlie     3.0   10.0
3            4    Diana     1.0    NaN
4            5      Eve     2.0    NaN
5            6      NaN     NaN   75.0


#### When to use which:

inner — you only want complete data,
left — your left table is the master, right enriches it,
outer — you want everything, deal with nulls later

## Part 4 - Exercises

In [38]:
#Load fresh copy for exercises
df = pd.read_csv(url)

In [39]:
# Exercise 1
# Handle ALL missing values in the dataset properly
# Age → fill with median
# Embarked → fill with mode
# Cabin → drop the column
# Verify with isnull().sum() at the end

median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age)

mode_embarked = df['Embarked'].mode()[0]
df['Embarked'] = df['Embarked'].fillna(mode_embarked)

df = df.drop(columns=['Cabin'])

print(df.isnull().sum())

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64


In [40]:
# Exercise 2
# Create a new column 'Title' extracted from the Name column
# Names look like: "Braund, Mr. Owen Harris"
# Hint: df['Name'].str.split(', ').str[1].str.split('.').str[0]
# Then do value_counts() to see all titles
# Then group rare titles (everything that isn't Mr/Mrs/Miss/Master) into 'Other'

df['Title'] = df['Name'].str.split(', ').str[1].str.split('.').str[0]

print(df['Title'].value_counts())

def group_titles(title):
    if title=='Mr':
        return 'Mr'
    elif title=='Mrs':
        return 'Mrs'
    elif title=='Miss':
        return 'Miss'
    elif title=='Master':
        return 'Master'
    else:
        return 'Other'

df['TitleGroup'] = df['Title'].apply(group_titles)
print(df['TitleGroup'].value_counts())

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64
TitleGroup
Mr        517
Miss      182
Mrs       125
Master     40
Other      27
Name: count, dtype: int64


In [20]:
# Exercise 3
# Create a 'FareCategory' column using pd.cut()
# Read the docs hint: pd.cut(df['Fare'], bins=3, labels=['Low','Medium','High'])
# Find survival rate per FareCategory

df['FareCategory'] = pd.cut(df['Fare'], bins=3, labels=['Low','Medium','High'])

print(df.groupby('FareCategory')['Survived'].mean())

FareCategory
Low       0.376579
Medium    0.647059
High      1.000000
Name: Survived, dtype: float64


C:\Users\mohiu\AppData\Local\Temp\ipykernel_24164\3468890815.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('FareCategory')['Survived'].mean())


In [52]:
# Exercise 4 (stretch)
# Create a final clean DataFrame with these columns only:
# Survived, Pclass, Sex_encoded, Age, Fare, IsAlone, FamilySize
# No missing values anywhere
# Print shape and first 5 rows
# This is the actual dataset you'll train your ML model on in Week 2
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df_clean = df[['Survived', 'Pclass', 'Sex_encoded', 'Age', 'Fare', 'IsAlone', 'FamilySize']]
print(df_clean.isnull().sum())
print(df_clean.shape)
print(df_clean.head())

Survived       0
Pclass         0
Sex_encoded    0
Age            0
Fare           0
IsAlone        0
FamilySize     0
dtype: int64
(891, 7)
   Survived  Pclass  Sex_encoded   Age     Fare IsAlone  FamilySize
0         0       3            0  22.0   7.2500      No           2
1         1       1            1  38.0  71.2833      No           2
2         1       3            1  26.0   7.9250     Yes           1
3         1       1            1  35.0  53.1000      No           2
4         0       3            0  35.0   8.0500     Yes           1


In [53]:
df_clean.to_csv('titanic_clean.csv', index=False)

### Reviews

Good. Ex1 is clean and correct.


#### Exercise 2 — Excellent ✅
The string chaining is correct and the `group_titles` function works perfectly. One thing to know — a more concise way to do the same thing that you'll see in real codebases:

```python
common_titles = ['Mr', 'Mrs', 'Miss', 'Master']
df['TitleGroup'] = df['Title'].apply(lambda x: x if x in common_titles else 'Other')
```
Your version is totally fine, especially for readability. Just know this pattern exists.

---

#### Exercise 3 — Correct but one thing to understand ⚠️
`pd.cut` with `bins=3` splits the Fare range into 3 **equal-width** intervals. Run this:
```python
print(df['Fare'].describe())
print(df['FareCategory'].value_counts())
```
You'll notice almost everyone falls in "Low" because a few people paid extremely high fares (outliers), making the bins very uneven. In real work you'd use:
```python
df['FareCategory'] = pd.qcut(df['Fare'], q=3, labels=['Low','Medium','High'])
```
`pd.qcut` splits into equal-**size** groups instead of equal-width. Remember this distinction — it matters for ML features.

---

### Exercise 4 — Almost, one bug 🐛
You referenced `Sex_encoded` and `IsAlone` but never created them in this notebook. You need to add before building `df_clean`:

```python
df['Sex_encoded'] = df['Sex'].map({'male': 0, 'female': 1})
df['IsAlone'] = df.apply(lambda row: 1 if (row['SibSp'] + row['Parch']) == 0 else 0, axis=1)
```
Then your `df_clean` creation is correct. After fixing, save it:
```python
df_clean.to_csv('titanic_clean.csv', index=False)
```
Keep this file — you're feeding it into your ML model on Day 8.

---

### Overall verdict
Days 1–3 done. You're handling real data competently. The instincts are right, the gaps are small details.

Fix Ex4, push the clean CSV

In [54]:
git add.

SyntaxError: invalid syntax (3081662196.py, line 1)